# Generate web search eval JSONL

Read prompt records from a selected `data/search_eval*.jsonl` file, call the Responses API with the `web_search` tool enabled, and write raw response payloads to `outputs/web_search_eval_raw_<dataset>_<timestamp>.jsonl`.

Required environment variables (load from `.env`):
- `AZURE_OPENAI_BASE_URL` (or `AZURE_OPENAI_API_BASE` / `AZURE_EXISTING_AIPROJECT_ENDPOINT`)
- `AZURE_OPENAI_MODEL` (or `AZURE_OPENAI_DEPLOYMENT`)

In [11]:
# Load standard libraries, dotenv, and OpenAI client helpers for the eval run.
import json
import os
import time
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from openai import APITimeoutError, APIConnectionError, InternalServerError, RateLimitError

dotenv_candidates = [Path("../.env"), Path("../../.env")]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path, override=True)
    print(f"Loaded env from {dotenv_path}")
else:
    load_dotenv(override=True)
    print("No local .env file found; using the current process environment.")

Loaded env from ..\.env


In [12]:
# Choose a dataset file, then read the evaluation prompts and ensure each record has a stable identifier.
data_dir = Path("../data")
available_data_paths = sorted(data_dir.glob("search_eval*.jsonl"))
available_dataset_names = [path.name for path in available_data_paths]
print(available_dataset_names)

# Change this to any filename shown above, or set SEARCH_EVAL_DATASET in the environment.
dataset_filename = os.getenv("SEARCH_EVAL_DATASET", "search_eval_iac_github.jsonl")
if dataset_filename not in available_dataset_names:
    raise ValueError(
        f"Dataset '{dataset_filename}' was not found. Available options: {available_dataset_names}"
    )

data_path = data_dir / dataset_filename
records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
for index, record in enumerate(records, start=1):
    record.setdefault("id", f"search_{index:03d}")

print(f"Using dataset: {data_path.name} ({len(records)} records)")
records[:3]

['search_eval.jsonl', 'search_eval_blocklist_domains_20250826.jsonl', 'search_eval_iac_github.jsonl', 'search_eval_jpmc.jsonl']
Using dataset: search_eval_iac_github.jsonl (5 records)


[{'id': 'search_iac_github_001',
  'query': 'From the git repo, branch FEATURE-apigee; github.com/abhijeettalele/gcp-sandbox/tree/FEATURE-apigee/iac/terraform/11mmgmt/modules/apigee/main.tf, can you pull the code sample for paving apigee?',
  'messages': [{'role': 'system',
    'content': 'You are a helpful AI/ML data engineer assistant that writes IaC code to pave services.\n\n- When the user asks for a code sample from a git repository, use the web_search tool to fetch the referenced file instead of guessing.\n- If the user provides a GitHub repo, branch, and file path, prefer opening the exact GitHub contents API URL for that file.\n- Return the relevant code sample or a concise summary if the file cannot be retrieved.\n- Answer directly and factually; do not include greetings or introductory text.\n'},
   {'role': 'user',
    'content': 'From the git repo, branch FEATURE-apigee; github.com/abhijeettalele/gcp-sandbox/tree/FEATURE-apigee/iac/terraform/11mmgmt/modules/apigee/main.tf, 

In [13]:
# Azure OpenAI client setup (uses managed identity and env vars).

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI
import os


token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
policy_id = os.getenv("AZURE_OPENAI_POLICY_ID", "customfilter")
credential = os.getenv("AZURE_OPENAI_API_KEY") or token_provider
client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=credential,
    default_headers={"x-policy-id": policy_id},
)

model = os.environ["AZURE_OPENAI_DEPLOYMENT"]


In [ ]:
# Call the Responses API with web search enabled and capture raw payloads.
sleep_seconds = 2.0
responses: list[dict[str, object]] = []

max_retries = 3
base_backoff_seconds = 2.0

for record in records:
    response = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model=model,
                tools=[{"type": "web_search"}],
                input=record.get("messages") or record["query"],
                reasoning={"effort": "none"}, #none
                extra_headers={"x-policy-id": policy_id},
            )
            break
        except (RateLimitError, InternalServerError, APIConnectionError, APITimeoutError) as exc:
            if attempt == max_retries:
                responses.append(
                    {
                        "id": record.get("id"),
                        "query": record.get("query"),
                        "test_case_description": record.get("test_case_description"),
                        "error": {
                            "type": type(exc).__name__,
                            "message": str(exc),
                            "attempts": attempt,
                        },
                    }
                )
            else:
                retry_wait = base_backoff_seconds * attempt
                time.sleep(retry_wait)
    if response is not None:
        responses.append(
            {
                "id": record.get("id"),
                "query": record.get("query"),
                "test_case_description": record.get("test_case_description"),
                "response": response.model_dump(),
            }
        )
    if sleep_seconds:
        time.sleep(sleep_seconds)

len(responses)

5

In [15]:
# Persist the raw response payloads as a JSONL log for evaluation.
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dataset_slug = data_path.stem.replace("search_eval_", "")
output_path = output_dir / f"web_search_eval_raw_{dataset_slug}_{timestamp}.jsonl"

payload = "\n".join(json.dumps(item, ensure_ascii=False) for item in responses) + "\n"
output_path.write_text(payload, encoding="utf-8")
output_path

WindowsPath('../outputs/web_search_eval_raw_iac_github_20260616_083547.jsonl')

In [16]:
# Quick sanity check: how many responses succeeded or failed.
total = len(responses)
error_count = sum(1 for item in responses if "error" in item)
(total, error_count)

(5, 4)